# **Recurrent Neural Network(RNN)**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding, LSTM, Dropout, Bidirectional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

text = 'This is GeeksforGeeks a software training institute'
chars = sorted(list(set(text)))
char_to_index = {char : i for i , char in enumerate(chars)}
index_to_char = {i : char for i , char in enumerate(chars)}

In [ ]:
seq_length = 3
sequences = []
labels = []

for i in range(len(text) - seq_length):
  seq = text[i:i+seq_length]
  label = text[i + seq_length]
  sequences.append([char_to_index[char] for char in seq])
  labels.append(char_to_index[label])

len(sequences), len(labels)

In [ ]:
from tensorflow.keras.layers import Input

model = Sequential([
    Input(shape=(seq_length, len(chars))),
    SimpleRNN(50, activation='relu'),
    Dense(len(chars), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
X_train_data = np.array(sequences)
y_train_data = np.array(labels)
X_train_data.shape, y_train_data.shape

In [ ]:
X_train_final = tf.one_hot(X_train_data, len(chars))
y_train_final = tf.one_hot(y_train_data, len(chars))
X_train_final.shape, y_train_final.shape

In [ ]:
print(f"Training on {X_train_final.shape[0]} samples...")
model.fit(X_train_final, y_train_final, epochs=500, verbose=0)
print("Model training complete.")

In [ ]:
start_seq = "This is G"
generated_text = start_seq

for i in range(50):
    current_seq = generated_text[-seq_length:]

    x_input = np.array([[char_to_index[char] for char in current_seq]])
    x_one_hot = tf.one_hot(x_input, len(chars))

    prediction = model.predict(x_one_hot, verbose=0)
    next_index = np.argmax(prediction)
    next_char = index_to_char[next_index]

    generated_text += next_char

print('Generated Text:')
print(generated_text)

# **Long Short-Term Memory Networks (LSTMs)**

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import spacy

text = """
Machine learning is amazing
Machine learning is powerful
Deep learning is part of machine learning
Artificial intelligence is the future
I love learning new things
Natural language processing is interesting
"""

nlp = spacy.load('en_core_web_sm')
doc = nlp(text.lower())
tokens = []

for token in doc:
  if (not token.is_stop and not token.is_punct and token.is_alpha):
    tokens.append(token.lemma_)

preprocess_text = ' '.join(tokens)
preprocess_text

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([preprocess_text])

total_words = len(tokenizer.word_index) + 1
total_words

In [ ]:
input_sequences = []

token_list = tokenizer.texts_to_sequences([preprocess_text])[0]
for i in range(1, len(token_list)):
  input_sequences.append(token_list[:i+1])

input_sequences

In [ ]:
max_len = max(len(seq) for seq in input_sequences)

input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_len, padding='pre'))

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = to_categorical(y, num_classes=total_words)

In [ ]:
model = Sequential([
    Embedding(total_words, 64, input_length=max_len - 1),
    LSTM(128),
    Dense(total_words, activation='softmax')
])

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.fit(X, y, epochs=200)

In [ ]:
def predict_word(seed_text):
  token_list = tokenizer.texts_to_sequences([seed_text])[0]
  token_list = pad_sequences([token_list], maxlen=max_len - 1, padding='pre')

  y_pred = model.predict(token_list, verbose=0)
  predict_index = np.argmax(y_pred)

  for word, index in tokenizer.word_index.items():
    if index == predict_index:
      return word

predict_word('machine')

In [ ]:
predict_word('learning')

# **Gated Recurrent Unit Networks**

In [ ]:
df = pd.read_csv('/content/data.csv', parse_dates=['Date'], index_col='Date')
df.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df.values)
scaled_data

In [ ]:
def create_sequences(data, time_steps):
  X, y = [], []

  for i in range(len(data) - time_steps - 1):
    X.append(data[i:(i + time_steps)])
    y.append(data[i + time_steps, 0])

  return np.array(X), np.array(y)

time_steps = 10
X, y = create_sequences(scaled_data, time_steps)
X = X.reshape(X.shape[0], X.shape[1], 1)
X.shape, y.shape

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam

model = Sequential()
model.add(GRU(units=50, return_sequences=True, input_shape=(X.shape[1], 1)))
model.add(GRU(units=50))
model.add(Dense(units=1))

model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
model.summary()

In [ ]:
model.fit(X, y, epochs=10, batch_size=32)

In [ ]:
input_seq = scaled_data[-time_steps:].reshape(1, time_steps, 1)
predicted_values = model.predict(input_seq)
predicted_values = scaler.inverse_transform(predicted_values)
predicted_values

In [ ]:
print(
    f"The predicted temperature for the next day is: {predicted_values[0][0]:.2f}°C")

# **Bidirectional Recurrent Neural Network**

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from tensorflow.keras.datasets import imdb
from tensorflow.keras.layers import Bidirectional

features = 2000
max_len = 50

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=features)

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [ ]:
embedding_dim = 128
hidden_units = 64

model = Sequential()
model.add(Embedding(input_dim=features, output_dim=embedding_dim, input_length=max_len))
model.add(Bidirectional(SimpleRNN(hidden_units)))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
batch_size = 32
epochs = 5

model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(X_test, y_test))

In [ ]:
from sklearn.metrics import classification_report

loss , accuracy = model.evaluate(X_test, y_test)
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5)

classification_report(y_test, y_pred, target_names=['Negative', 'Positive'])